Notebook to place a pattern on a FIB image

In [ ]:
# general
from __future__ import annotations

import logging
import re
import tempfile
import typing
from dataclasses import dataclass, field
from importlib import resources
from os import PathLike
from pathlib import Path

import matplotlib.pyplot as plt
import tifffile
import yaml

# Set up test microscope
from fibsem import utils

# for the fibsem structures
from fibsem.milling import get_milling_stages
from fibsem.milling.patterning.plotting import draw_milling_patterns
from fibsem.structures import ImageSettings, Point

from adaptive_milling._dataclasses import CycleInformation, CycleTimestamps

# Adaptive polishing
from adaptive_milling.strategy import BitmapAdaptivePolishMillingStrategy

if typing.TYPE_CHECKING:
    from fibsem.milling import FibsemMillingStage
    from fibsem.structures import FibsemImage
    from numpy.typing import NDArray


class NoImageFound(FileNotFoundError):
    pass


class NoExperimentFound(NoImageFound):
    pass


_logger = logging.getLogger(__name__)

In [ ]:
base_path = (
    Path.home()
    / "OneDrive - The Rosalind Franklin Institute"
    / "Documents"
    / "test data"
    / "adaptive milling"
)

model_path = (
    base_path
    / "sem_models"
    / "Gen1"
    / "gen01_quality_1536_v9_FPN"
    / "cryo_sem_epoch_73.pth"
)
model_generation = "1.0"

experiment_paths = (
    base_path / "20251027_AM_bitmaptesting" / "G4" / "AutoLamella-2025-10-27-10-17",
    base_path / "20251027_AM_bitmaptesting" / "G4" / "AutoLamella-2025-10-27-14-53",
    base_path / "20251027_AM_bitmaptesting" / "G4" / "AutoLamella-2025-10-27-16-58",
    base_path / "20251027_AM_bitmaptesting" / "G4" / "AutoLamella-2025-10-27-18-27",
)

# Path to the microscope configuration that will be used
microscope_config_path = (
    Path(str(resources.files("fibsem"))) / "config" / "microscope-configuration.yaml"
).resolve()

In [ ]:
def setup_test_protocol(
    protocol_template: dict[str, typing.Any] | str | PathLike[str],
    temporary_directory: Path,
    model_path: str | PathLike[str],
    model_generation: str,
    **protocol_kwargs: typing.Any,
) -> Path:
    """Sets up a demo microscope"""
    if isinstance(protocol_template, (str, PathLike)):
        with Path(protocol_template).open() as f:
            protocol_template_dict = yaml.safe_load(f)
    else:
        protocol_template_dict = protocol_template

    # Rough 1 to bitmap
    rough_milling_stage = protocol_template_dict["milling"]["mill_rough"][0]
    rough_milling_stage["strategy"]["name"] = "BitmapAdaptivePolishing"
    rough_milling_stage["strategy"]["config"].update(
        {
            "model_path": str(model_path),
            "model_generation": model_generation,
            **protocol_kwargs,
        }
    )

    # AP to bitmap
    for i, polishing_stage in enumerate(
        protocol_template_dict["milling"]["mill_polishing"]
    ):
        if polishing_stage["strategy"]["name"] in (
            "AdaptivePolishing",
            "BitmapAdaptivePolishing",
        ):
            break
    polishing_stage["strategy"]["name"] = "BitmapAdaptivePolishing"

    # Update keys that have been renamed
    updated_keys = [
        ("gis_stop_um", "gis_stop_min"),
        ("gis_stop_min_um", "gis_stop_min"),
        ("gis_stop_median_um ", "gis_stop_median"),
        ("max_crack_area_um2 ", "max_crack_area"),
        ("minimum_lamella_area_um2 ", "minimum_lamella_area"),
        ("maximum_drift_um ", "maximum_drift"),
        ("gis_max_um ", "gis_max"),
        ("gis_min_um ", "gis_min"),
    ]
    for old_key, new_key in updated_keys:
        if old_key in polishing_stage["strategy"]["config"]:
            value = polishing_stage["strategy"]["config"].pop(old_key)
            if old_key.endswith("_um"):
                value *= 1e-6
            elif old_key.endswith("_um2"):
                value *= 1e-12

            polishing_stage["strategy"]["config"][new_key] = value

    # Ensure new defaults don't override old behaviour
    if "gis_stop_median" not in polishing_stage["strategy"]["config"]:
        polishing_stage["strategy"]["config"]["gis_stop_median"] = 0
    if "apply_boundary_smoothing" not in polishing_stage["strategy"]["config"]:
        polishing_stage["strategy"]["config"]["apply_boundary_smoothing"] = False
    if "incident_angle_sputter_ratio" not in polishing_stage["strategy"]["config"]:
        polishing_stage["strategy"]["config"]["incident_angle_sputter_ratio"] = 1

    polishing_stage["strategy"]["config"].update(
        {
            "model_path": str(model_path),
            "model_generation": model_generation,
            **protocol_kwargs,
        }
    )
    protocol_template_dict["milling"]["mill_polishing"][i] = polishing_stage

    protocol_path = temporary_directory / "protocol.yaml"
    with protocol_path.open("w") as f:
        yaml.safe_dump(protocol_template_dict, f)

    return protocol_path

In [ ]:
def get_fib_and_sem_directories(
    data_directory: str | PathLike[str],
) -> tuple[Path, Path]:
    data_directory = Path(data_directory)
    if not data_directory.is_dir():
        raise FileNotFoundError(f"{data_directory} could not be found")
    fib_image_dir = data_directory / "fib"
    sem_image_dir = data_directory / "sem"
    if not fib_image_dir.is_dir():
        fib_image_dir = data_directory / "FIB"
        if not fib_image_dir.is_dir():
            raise FileNotFoundError("Failed to find FIB directory")
    if not sem_image_dir.is_dir():
        sem_image_dir = data_directory / "SEM"
        if not sem_image_dir.is_dir():
            raise FileNotFoundError("Failed to find SEM directory")
    return (fib_image_dir, sem_image_dir)

In [ ]:
def run_placement(
    cycle: int,
    milling_stages: list[FibsemMillingStage],
    sem_image: FibsemImage,
    fib_image: FibsemImage,
    plots_directory: str | PathLike[str],
) -> None:
    updated_milling_stages: list[FibsemMillingStage] = []
    for i, milling_stage in enumerate(milling_stages):
        strategy = typing.cast(
            BitmapAdaptivePolishMillingStrategy, milling_stage.strategy
        )
        if strategy.model is None:
            strategy._load_model()
            _logger.info("Loaded model")

        assert sem_image.metadata is not None

        _logger.info("Getting lamella info")
        cycle_info = CycleInformation(
            milling_cycle=cycle,
            identifier=f"{sem_image.metadata.image_settings.filename}_{i}",
            timestamps=CycleTimestamps(),
        )
        lamella_info = strategy._get_lamella_info(
            cycle_info=cycle_info,
            fib_image=fib_image,
            sem_image=sem_image,
            lamella_pad_x=strategy.config.lamella_pad_x,
        )

        _logger.info("Updating milling stage")
        # Make a copy of the milling stage before updating the pattern
        updated_milling_stage = strategy._update_milling_stage(
            stage=milling_stage, lamella_info=lamella_info
        )
        updated_milling_stages.append(updated_milling_stage)

        # TODO: Try using AnnotationBbox and OffsetImage for bitmap plotting (current method is too slow for large images)
        _logger.info("Creating stage plot")
        strategy._create_milling_cycle_plot(
            plots_directory=Path(plots_directory),
            lamella_info=lamella_info,
            stage=updated_milling_stage,
        )

    if updated_milling_stages:
        _logger.info("Creating multistage plot")
        fig, _ = draw_milling_patterns(
            fib_image,
            updated_milling_stages,
            crosshair=False,
            highlight_overlaps=True,
            ax=None,
        )
        figure_path = (
            Path(plots_directory)
            / f"{lamella_info.identifier}_{cycle}_bitmap_pattern_plot.tif"
        )
        print(f'Saving figure to "{figure_path}"')
        fig.savefig(figure_path)
        plt.close("all")

In [ ]:
@dataclass
class Metadata:
    pixel_size: Point
    image_settings: ImageSettings


@dataclass
class TestImageInfo:
    path: Path
    data: NDArray[typing.Any]
    pixel_size_m: float
    metadata: Metadata = field(init=False)

    def __post_init__(self) -> None:
        self.metadata = Metadata(
            pixel_size=Point(x=self.pixel_size_m, y=self.pixel_size_m),
            image_settings=ImageSettings(
                hfw=4e-5,
                resolution=(self.data.shape[0], self.data.shape[1]),
                path=str(self.path.parents),
                filename=self.path.stem,
            ),
        )

In [ ]:
def get_array_and_pixel_size(image_path: Path) -> tuple[NDArray[typing.Any], float]:
    with tifffile.TiffFile(image_path) as tif:
        image_array = tif.asarray()
        if tif.shaped_metadata is None:
            raise ValueError("Failed to read pixel size as shaped_metadata is None")
        return image_array, tif.shaped_metadata[0]["pixel_size"]["x"]

In [ ]:
def process_lamella(
    fib_path: Path,
    sem_path: Path,
    plot_dir: Path,
    milling_stages: list[FibsemMillingStage],
) -> None:
    image_regex = re.compile(r"^([\w_\-\. ]+)_img_(\d{3})_(?:SEM|FIB)$", re.IGNORECASE)
    for i, sem_image_path in enumerate(sem_path.glob("*.tif"), 1):
        m = image_regex.match(sem_image_path.stem)
        if m is None:
            raise NoImageFound(f"'{sem_image_path.stem}' is not a valid image name")

        fib_image_path = fib_path / f"{m.group(1)}_img_{m.group(2)}_FIB.tif"
        sem_info = TestImageInfo(
            sem_image_path, *get_array_and_pixel_size(sem_image_path)
        )
        fib_info = TestImageInfo(
            fib_image_path, *get_array_and_pixel_size(fib_image_path)
        )
        _logger.info("Loaded images")
        run_placement(
            i,
            milling_stages=milling_stages,
            sem_image=sem_info,  # type: ignore
            fib_image=fib_info,  # type: ignore
            plots_directory=plot_dir,
        )

In [ ]:
def process_experiment(
    experiment_path: str | PathLike[str],
    plots_directory: str | PathLike[str],
    include_rough: bool = False,
    **protocol_kwargs: typing.Any,
) -> None:
    experiment_path = Path(experiment_path)
    plots_directory = Path(plots_directory)
    plots_directory.mkdir(exist_ok=True)
    experiment_yaml_path = experiment_path / "experiment.yaml"
    protocol_path = experiment_path / "protocol.yaml"
    assert protocol_path.is_file()

    with experiment_yaml_path.open() as f:
        experiment_dict = yaml.safe_load(f)

    experiment_positions = experiment_dict["positions"]

    for position in experiment_positions:
        lamella_name = position["petname"]
        expected_path = experiment_path / lamella_name
        if expected_path.is_dir():
            print(f"Starting pattern placement for {lamella_name}")
            with tempfile.TemporaryDirectory() as tmp_dir:
                protocol_path = setup_test_protocol(
                    {"milling": position["protocol"]},
                    temporary_directory=Path(tmp_dir),
                    model_path=model_path,
                    model_generation=model_generation,
                    **protocol_kwargs,
                )
                # Set up test microscope
                _, settings = utils.setup_session(
                    config_path=microscope_config_path, protocol_path=protocol_path
                )
            assert settings.protocol is not None

            milling_stages: list[FibsemMillingStage] = []

            if include_rough:
                rough_milling_stages = get_milling_stages(
                    "mill_rough", settings.protocol["milling"]
                )
                for rough_stage in rough_milling_stages:
                    if rough_stage.strategy.name == "BitmapAdaptivePolishing":
                        milling_stages.append(rough_stage)
                        break

            polishing_stages = get_milling_stages(
                "mill_polishing", settings.protocol["milling"]
            )
            for polishing_stage in polishing_stages[::-1]:
                if polishing_stage.strategy.name == "BitmapAdaptivePolishing":
                    milling_stages.append(polishing_stage)
                    break

            ap_subdirs = list(expected_path.glob("*adaptive_polish_*"))
            if ap_subdirs:
                for ap_dir in ap_subdirs:
                    print(
                        f"Starting pattern placement for {lamella_name}/{ap_dir.name}"
                    )
                    try:
                        fib_image_dir, sem_image_dir = get_fib_and_sem_directories(
                            ap_dir
                        )
                        plot_subdir = plots_directory / expected_path.name / ap_dir.name
                        plot_subdir.mkdir(exist_ok=True, parents=True)

                        process_lamella(
                            fib_path=fib_image_dir,
                            sem_path=sem_image_dir,
                            plot_dir=plot_subdir,
                            milling_stages=milling_stages,
                        )
                    except FileNotFoundError:
                        continue

In [ ]:
for experiment_path in experiment_paths:
    plots_directory = experiment_path / "bitmap_pattern_plots"
    try:
        process_experiment(
            experiment_path,
            plots_directory=plots_directory,
            # apply_boundary_smoothing=False,
            # incident_angle_scaling=True,
            # bitmap_gaussian_sigma=20,
            # bitmap_erosion_px=40,
            # apply_boundary_smoothing=True,
            # lamella_gis_boundary_gaussian_sigma=15,
        )
    except Exception:
        _logger.exception('Stopped experiment "%s"', experiment_path)